# Audyt danych

In [15]:
from pathlib import Path
import pandas as pd

file = 'product+classification+and+clustering.zip'
folders = [Path('data'), Path('../data'), Path('data/sample'), Path('../data/sample')]
path = next((folder / file for folder in folders if (folder / file).exists()), None)

if path is None:
    raise FileNotFoundError('Nie znaleziono pliku ZIP w lokalnym folderze data/')

df = pd.read_csv(path, compression='zip')
df.columns = df.columns.str.strip()

print('Wymiary zbioru:', df.shape)
print('Kolumny:', df.columns.tolist())

Wymiary zbioru: (35311, 7)
Kolumny: ['Product ID', 'Product Title', 'Merchant ID', 'Cluster ID', 'Cluster Label', 'Category ID', 'Category Label']


## Podstawowe informacje i rozkład klas

In [16]:
print('Liczba ofert:', len(df))
print('Liczba produktów:', df['Cluster ID'].nunique())
print('Liczba kategorii:', df['Category Label'].nunique())
print('Liczba sprzedawców:', df['Merchant ID'].nunique())

print('\nRozkład klas:')
print(df['Category Label'].value_counts())

Liczba ofert: 35311
Liczba produktów: 13233
Liczba kategorii: 10
Liczba sprzedawców: 306

Rozkład klas:
Category Label
Fridge Freezers     5501
Mobile Phones       4081
Washing Machines    4044
CPUs                3862
Fridges             3584
TVs                 3564
Dishwashers         3424
Digital Cameras     2697
Microwaves          2342
Freezers            2212
Name: count, dtype: int64


## Braki danych i duplikaty

In [17]:
print('Braki danych:')
print(df.isna().sum())

print('\nZduplikowane wiersze:', df.duplicated().sum())
print('Powtórzone tytuły:', df['Product Title'].duplicated().sum())

titles = df.groupby('Product Title')['Cluster ID'].nunique()
print('Tytuły w różnych klastrach:', (titles > 1).sum())

labels = df.groupby('Cluster ID')['Category Label'].nunique()
print('Klastry z różnymi kategoriami:', (labels > 1).sum())

conflicts = df.groupby("Product Title")["Category Label"].nunique()
print("Tytuły z różnymi kategoriami:", (conflicts > 1).sum())

Braki danych:
Product ID        0
Product Title     0
Merchant ID       0
Cluster ID        0
Cluster Label     0
Category ID       0
Category Label    0
dtype: int64

Zduplikowane wiersze: 0
Powtórzone tytuły: 4318
Tytuły w różnych klastrach: 617
Klastry z różnymi kategoriami: 0
Tytuły z różnymi kategoriami: 2


## Długości tekstów i wielkości grup

In [18]:
print('Długość tytułów w znakach:')
print(df['Product Title'].str.len().describe())

print('\nDługość tytułów w słowach:')
print(df['Product Title'].str.split().str.len().describe())

print('\nLiczba ofert na produkt:')
print(df.groupby('Cluster ID').size().describe())

Długość tytułów w znakach:
count    35311.000000
mean        52.982215
std         22.447138
min          5.000000
25%         38.000000
50%         52.000000
75%         65.000000
max        200.000000
Name: Product Title, dtype: float64

Długość tytułów w słowach:
count    35311.000000
mean         8.665911
std          4.166677
min          1.000000
25%          6.000000
50%          8.000000
75%         11.000000
max         37.000000
Name: Product Title, dtype: float64

Liczba ofert na produkt:
count    13233.000000
mean         2.668405
std          2.473239
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         27.000000
dtype: float64


## Przykładowe rekordy

In [19]:
print(df[['Product ID', 'Product Title', 'Cluster ID', 'Category Label']].head(5))

   Product ID                                      Product Title  Cluster ID  \
0           1                    apple iphone 8 plus 64gb silver           1   
1           2                apple iphone 8 plus 64 gb spacegrau           1   
2           3  apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim...           1   
3           4                apple iphone 8 plus 64gb space grey           1   
4           5  apple iphone 8 plus gold 5.5 64gb 4g unlocked ...           1   

  Category Label  
0  Mobile Phones  
1  Mobile Phones  
2  Mobile Phones  
3  Mobile Phones  
4  Mobile Phones  


## Grupowanie produktów

In [20]:
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42

parent = {x: x for x in df["Cluster ID"].unique()}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

# Łączenie klastrów, które mają identyczne tytuły

normalized_titles = df["Product Title"].str.lower().str.replace(
    r"[^a-z0-9]", "", regex=True
)

for _, group in df.groupby(normalized_titles):
    ids = group["Cluster ID"].unique()
    for x in ids[1:]:
        parent[find(x)] = find(ids[0])

df["split_group"] = df["Cluster ID"].map(find)

print("Liczba grup:", df["split_group"].nunique())

Liczba grup: 11967


## Podział i zapis plików

In [21]:
splitter = StratifiedGroupKFold(
    n_splits=10,
    shuffle=True,
    random_state=SEED
)

folds = list(splitter.split(
    df, df["Category Label"], df["split_group"]
))

val_idx = folds[1][1]
test_idx = folds[2][1]
train_idx = df.index.difference(val_idx).difference(test_idx)

parts = {
    "train": df.loc[train_idx],
    "validation": df.loc[val_idx],
    "test": df.loc[test_idx]
}

# Sprawdzenie podziału
for name, part in parts.items():
    print(name, len(part), part["Cluster ID"].nunique())

for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
]:
    assert set(parts[a]["Cluster ID"]).isdisjoint(parts[b]["Cluster ID"])
    assert set(parts[a]["Product Title"]).isdisjoint(parts[b]["Product Title"])

assert sum(len(part) for part in parts.values()) == len(df)

print("\nRozkład klas w procentach:")
print(pd.DataFrame({
    name: (part["Category Label"].value_counts(normalize=True) * 100).round(1)
    for name, part in parts.items()
}))

for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    assert set(normalized_titles.loc[parts[a].index]).isdisjoint(
        set(normalized_titles.loc[parts[b].index])
    )

# Zapis identyfikatorów
split_dir = Path("splits") if Path("splits").is_dir() else Path("../splits")

for name, part in parts.items():
    part[["Product ID", "Cluster ID"]].rename(
        columns={"Product ID": "id", "Cluster ID": "group_id"}
    ).to_csv(split_dir / f"{name}_ids.csv", index=False)

print("Pliki CSV zapisane.")

train 28338 10684
validation 3479 1282
test 3494 1267

Rozkład klas w procentach:
                  train  validation  test
Category Label                           
CPUs               10.9        11.1  11.0
Digital Cameras     7.6         7.8   7.7
Dishwashers         9.6         9.8  10.4
Freezers            6.2         6.4   6.3
Fridge Freezers    15.5        15.8  15.7
Fridges            10.1        10.3  10.2
Microwaves          6.6         6.8   6.7
Mobile Phones      11.5        11.7  11.7
TVs                10.1        10.2  10.2
Washing Machines   11.8        10.1  10.0
Pliki CSV zapisane.
